# Data Cleaning

Reads the raw scraped CSVs in `data/` and writes cleaned versions to `data/cleaned/` for `main.ipynb`.

**Changes from the class version**
- Duplicates are removed by ID (book, reviewer, movie URL), not by exact row match. The old method left 789 duplicate books and about 5,200 duplicate reviews.
- Reviews link to their book or movie by ID or URL, not by title. 21 Goodreads titles belong to more than one book.
- Accented letters are kept as plain letters ("cliché" → "cliche"). Before, they were deleted, which created junk words like "clich".
- Critic ratings above the scale (`7.5/5`, `29/10`) are set to missing instead of producing scores above 1.
- All review ratings are rescaled so the lowest possible rating is 0 and the highest is 1.
- Film fields needed for the critic–audience gap work (release date, runtime, content rating, top-critic score) are kept.
- The cleaning functions are a tested Python package (`src/story_reception/`), imported here.
- Only the needed columns are loaded, text is cleaned with vectorized pandas string methods, and checks at the end fail loudly if anything is off.
- Each review gets a detected `language` (Lingua) in place of the old common-English-word heuristic.
- RT audience reviews keep a hashed `reviewer_id` (from the display name) so `main.ipynb` can check for reviewer leakage.
- Letterboxd is out of scope and no longer loaded.

In [1]:
import hashlib
import os
from collections import Counter

import numpy as np
import pandas as pd

# Cleaning helpers live in src/story_reception (tested in tests/); install with `pip install -e .`
from story_reception.clean import (clean_review_text, normalize_title, parse_critic_rating, parse_genres,
                                   runtime_to_minutes)
from story_reception.language import detect_language

pd.set_option('display.max_columns', None)

DATA_DIR = 'data'
CLEAN_DIR = os.path.join(DATA_DIR, 'cleaned')
MIN_WORDS = 5  # reviews shorter than this are mostly noise ("Loved it!!")

## 1. Load raw data

Only the columns used downstream are loaded. The Goodreads reviews file is about 370 MB, mostly from columns like `shelves` that we never use.

In [2]:
BOOK_COLS = ['url', 'book_id', 'title', 'title_complete', 'author', 'description',
             'avg_rating', 'ratings_count', 'reviews_count',
             'rating_dist_1', 'rating_dist_2', 'rating_dist_3', 'rating_dist_4', 'rating_dist_5',
             'publication_date', 'genres']
GR_REVIEW_COLS = ['book_url', 'book_title', 'reviewer_id', 'rating', 'review_text']
RT_MOVIE_COLS = ['url', 'title', 'genres', 'content_rating', 'release_date', 'runtime', 'original_language',
                 'tomatometer', 'tomatometer_count', 'critic_avg_rating', 'critic_sentiment',
                 'top_critic_score', 'top_critic_count',
                 'audience_score', 'audience_avg_rating', 'audience_review_count', 'audience_sentiment',
                 'audience_all_score', 'audience_all_avg_rating', 'audience_all_review_count',
                 'audience_all_sentiment', 'synopsis', 'critics_consensus', 'audience_consensus']
RT_AUDIENCE_COLS = ['movie_url', 'movie_title', 'user_name', 'score', 'review_text', 'verified']
RT_CRITIC_COLS = ['movie_url', 'movie_title', 'critic_name', 'publication', 'sentiment', 'rating', 'review_text']

def load(name, cols):
    return pd.read_csv(os.path.join(DATA_DIR, name), usecols=cols)

gr_books = load('goodreads-books.csv', BOOK_COLS).rename(columns={'url': 'book_url'})
gr_reviews = load('goodreads-reviews.csv', GR_REVIEW_COLS)
rt_movies = load('rotten-tomatoes-movies.csv', RT_MOVIE_COLS).rename(columns={'url': 'movie_url'})
rt_audience = load('rotten-tomatoes-audience-reviews.csv', RT_AUDIENCE_COLS)
rt_critics = load('rotten-tomatoes-critic-reviews.csv', RT_CRITIC_COLS)

raw_counts = {
    'Goodreads books': len(gr_books),
    'Goodreads reviews': len(gr_reviews),
    'RT movies': len(rt_movies),
    'RT audience reviews': len(rt_audience),
    'RT critic reviews': len(rt_critics),
}
for name, n in raw_counts.items():
    print(f"{name:22s} {n:>9,}")
print(f"{'Total':22s} {sum(raw_counts.values()):>9,}")

Goodreads books            6,330
Goodreads reviews        177,261
RT movies                  3,001
RT audience reviews       15,988
RT critic reviews         25,581
Total                    228,161


## 2. Remove duplicates by key

A book listed on several Goodreads lists was scraped more than once, and the copies differ slightly (rating counts change between scrapes, and the list name differs). An exact-row match misses those copies, so we dedupe on IDs instead:

| Table | One row per |
|---|---|
| Goodreads books | `book_id` (keep the copy with the most ratings, i.e. the latest scrape) |
| Goodreads reviews | `book_id` + `reviewer_id` |
| RT movies | `movie_url` |
| RT audience reviews | `movie_url` + `user_name` + `review_text` |
| RT critic reviews | `movie_url` + `critic_name` + `review_text` |

Reviews get a `book_id` through their book URL, so two different books with the same title never share reviews.

In [3]:
def report(label, before, after):
    print(f"{label:22s} {before:>9,} -> {after:>9,}  (dropped {before - after:,})")

# Map every scraped book URL to its book_id before collapsing books
url_to_book_id = gr_books.drop_duplicates('book_url').set_index('book_url')['book_id']

n = len(gr_books)
gr_books = (gr_books.dropna(subset=['book_id', 'avg_rating'])
                    .sort_values('ratings_count', ascending=False)
                    .drop_duplicates('book_id')
                    .sort_index())
report('Goodreads books', n, len(gr_books))

n = len(gr_reviews)
gr_reviews['book_id'] = gr_reviews['book_url'].map(url_to_book_id)
gr_reviews = (gr_reviews[gr_reviews['book_id'].isin(gr_books['book_id'])]
                        .drop_duplicates(['book_id', 'reviewer_id'])
                        .drop(columns='book_url'))
report('Goodreads reviews', n, len(gr_reviews))

n = len(rt_movies)
rt_movies = rt_movies.drop_duplicates('movie_url')
report('RT movies', n, len(rt_movies))

def dedupe_rt_reviews(df, label, author_col):
    n = len(df)
    df = (df[df['movie_url'].isin(rt_movies['movie_url'])]
            .drop_duplicates(['movie_url', author_col, 'review_text']))
    report(label, n, len(df))
    return df

rt_audience = dedupe_rt_reviews(rt_audience, 'RT audience reviews', 'user_name')
rt_critics = dedupe_rt_reviews(rt_critics, 'RT critic reviews', 'critic_name')

# Replace audience display names ("Kyle M") with a short hash, so main.ipynb can tell repeat reviewers apart
# without storing names. Names are first name + initial, so two people can share one ID.
rt_audience['reviewer_id'] = rt_audience['user_name'].fillna('').map(
    lambda name: hashlib.sha1(name.encode()).hexdigest()[:12] if name else np.nan)
rt_audience = rt_audience.drop(columns='user_name')

Goodreads books            6,330 ->     4,997  (dropped 1,333)


Goodreads reviews        177,261 ->   138,830  (dropped 38,431)
RT movies                  3,001 ->     3,001  (dropped 0)
RT audience reviews       15,988 ->    15,986  (dropped 2)
RT critic reviews         25,581 ->    25,570  (dropped 11)


## 3. Fix types and missing values

Missing text becomes an empty string. Missing numbers and dates stay missing (`NaN`) instead of being filled with `'Unknown'` or `0`, so later analysis can decide how to handle them.

In [4]:
# Goodreads books
gr_books['title'] = gr_books['title'].str.strip()
gr_books['title_complete'] = gr_books['title_complete'].fillna(gr_books['title']).str.strip()
gr_books[['author', 'description', 'genres']] = gr_books[['author', 'description', 'genres']].fillna('')
gr_books['reviews_count'] = gr_books['reviews_count'].fillna(0).astype(int)
gr_books['publication_date'] = pd.to_datetime(gr_books['publication_date'], errors='coerce')
gr_books['publication_year'] = gr_books['publication_date'].dt.year.astype('Int64')

# RT movies
rt_movies['title'] = rt_movies['title'].str.strip()
text_cols = ['genres', 'synopsis', 'critics_consensus', 'audience_consensus']
rt_movies[text_cols] = rt_movies[text_cols].fillna('')
rt_movies['release_date'] = pd.to_datetime(rt_movies['release_date'], errors='coerce')
rt_movies['release_year'] = rt_movies['release_date'].dt.year.astype('Int64')
rt_movies['runtime_min'] = runtime_to_minutes(rt_movies['runtime'].fillna('')).astype('Int64')
rt_movies = rt_movies.drop(columns='runtime')

print('Goodreads books missing publication year:', gr_books['publication_year'].isna().sum())
print('RT movies missing release year:          ', rt_movies['release_year'].isna().sum())
print('RT movies missing runtime:               ', rt_movies['runtime_min'].isna().sum())
print('RT movies missing critic / audience score:',
      rt_movies['tomatometer'].isna().sum(), '/', rt_movies['audience_score'].isna().sum())

Goodreads books missing publication year: 767
RT movies missing release year:           323
RT movies missing runtime:                27
RT movies missing critic / audience score: 427 / 288


## 4. Detect review language

`detect_language` is in `src/story_reception/language.py`.


Each review gets a `language` code (`en`, `es`, ...) from [Lingua](https://github.com/pemistahl/lingua-py), limited to the 30 languages that appear in this data so English isn't mistaken for a rare language. Detection runs on the first 400 characters of the raw text (HTML and URLs removed), before accents and non-Latin scripts are stripped in the next step. `main.ipynb` keeps only `language == 'en'` for the text models. This replaces the old common-word-ratio heuristic.

In [5]:
for label, df in [('Goodreads', gr_reviews), ('RT audience', rt_audience), ('RT critics', rt_critics)]:
    df['language'] = detect_language(df['review_text'])
    top = df['language'].value_counts().head(5)
    print(f"{label:12s} English {(df['language'] == 'en').mean():.1%}   top: " +
          ', '.join(f'{k} {v:,}' for k, v in top.items()))

Goodreads    English 91.5%   top: en 127,040, es 2,658, fa 1,625, ar 1,551, it 806


RT audience  English 96.3%   top: en 15,402, pt 224, es 186, it 36, tl 31


RT critics   English 99.2%   top: en 25,367, unknown 136, tl 14, fr 12, de 8


## 5. Clean review text

`clean_review_text` is in `src/story_reception/clean.py`; `tests/test_clean.py` checks each step.


Steps, run as vectorized pandas string operations over the whole column:
1. Remove HTML tags and URLs.
2. Fix the few mojibake sequences (`â€™` → `'`) and smart quotes, dashes and ellipses.
3. Turn accented letters into plain ones (`cliché` → `cliche`, `Brontë` → `Bronte`) instead of deleting them.
4. Remove whatever is still non-ASCII (emoji, symbols, other scripts).
5. Collapse whitespace.

In [6]:
example = pd.Series(["A cliché-ridden, naïve plot — Brontë it isn’t \U0001F644 https://t.co/x"])
print('before:', example[0])
print('after: ', clean_review_text(example)[0])

for df in (gr_reviews, rt_audience, rt_critics):
    df['review_text'] = clean_review_text(df['review_text'])
for col in ('synopsis', 'critics_consensus', 'audience_consensus'):
    rt_movies[col] = clean_review_text(rt_movies[col])
gr_books['description'] = clean_review_text(gr_books['description'])

before: A cliché-ridden, naïve plot — Brontë it isn’t 🙄 https://t.co/x
after:  A cliche-ridden, naive plot - Bronte it isn't


## 6. Drop short reviews

Reviews under `MIN_WORDS` words carry almost no language signal. This also drops reviews with no text.

In [7]:
def drop_short_reviews(df, label):
    df = df.assign(word_count=df['review_text'].str.count(r'\S+'))
    short = df['word_count'] < MIN_WORDS
    print(f"{label:12s} median words {df['word_count'].median():>4.0f}  mean {df['word_count'].mean():>6.1f}  "
          f"under {MIN_WORDS} words: {short.sum():>6,}  kept {(~short).sum():,}")
    return df[~short]

gr_reviews = drop_short_reviews(gr_reviews, 'Goodreads')
rt_audience = drop_short_reviews(rt_audience, 'RT audience')
rt_critics = drop_short_reviews(rt_critics, 'RT critics')

Goodreads    median words  186  mean  300.8  under 5 words:  3,469  kept 135,361
RT audience  median words   24  mean   44.7  under 5 words:    722  kept 15,264


RT critics   median words   25  mean   24.6  under 5 words:    434  kept 25,136


## 7. Parse and rescale ratings

`parse_critic_rating` is in `src/story_reception/clean.py`; `tests/test_clean.py` covers every format below.


RT critic ratings come in 15+ formats (`3/5`, `B+`, `3 1/2 stars`, `7/10`, ...). `parse_critic_rating` converts each to a 0–5 score and returns `NaN` for anything it can't read or that falls outside 0–5 (for example `7.5/5`).

Then every review rating is rescaled to `rating_normalized` with the min-max formula (rating − lowest possible) / (highest − lowest), so the lowest rating on each platform is 0 and the highest is 1:

| Source | Raw scale | Formula |
|---|---|---|
| Goodreads | 1–5 stars | (rating − 1) / 4 |
| RT audience | 0.5–5 stars | (score − 0.5) / 4.5 |
| RT critics | parsed 0–5 | rating / 5 |

About 38% of critic reviews have no numeric rating but do have a fresh/rotten verdict, kept as `is_positive`.

In [8]:
# Parse each distinct rating string once, then map back
has_rating = rt_critics['rating'].notna()
parsed_lookup = {r: parse_critic_rating(r) for r in rt_critics.loc[has_rating, 'rating'].unique()}
rt_critics['rating_parsed'] = rt_critics['rating'].map(parsed_lookup)

unparsed = rt_critics.loc[has_rating & rt_critics['rating_parsed'].isna(), 'rating']
print(f"RT critic ratings: {len(rt_critics):,} reviews, {has_rating.sum():,} with a rating, "
      f"{rt_critics['rating_parsed'].notna().sum():,} parsed")
print(f"Set to NaN (unreadable or off-scale): {len(unparsed)} -> {unparsed.value_counts().to_dict()}")

rt_critics['is_positive'] = (rt_critics['sentiment'] == 'positive').astype(int)

gr_reviews['rating_normalized'] = (gr_reviews['rating'] - 1) / 4
rt_audience['rating_normalized'] = (rt_audience['score'] - 0.5) / 4.5
rt_critics['rating_normalized'] = rt_critics['rating_parsed'] / 5

gr_reviews['source'] = 'goodreads'
rt_audience['source'] = 'rt_audience'
rt_critics['source'] = 'rt_critic'

review_tables = {'Goodreads': gr_reviews, 'RT audience': rt_audience, 'RT critics': rt_critics}

print()
for label, df in review_tables.items():
    col = df['rating_normalized']
    print(f"{label:12s} range {col.min():.2f}-{col.max():.2f}  mean {col.mean():.3f}  missing {col.isna().sum():,}")

RT critic ratings: 25,136 reviews, 15,616 with a rating, 15,612 parsed
Set to NaN (unreadable or off-scale): 4 -> {'7.5/5': 2, '29/10': 1, '8.5/5': 1}

Goodreads    range 0.00-1.00  mean 0.738  missing 3,663
RT audience  range 0.00-1.00  mean 0.673  missing 0
RT critics   range 0.00-1.00  mean 0.639  missing 9,524


## 8. Titles and genres

`title_join` (lowercase, no leading article) is kept for matching titles across platforms later, such as books to their film adaptations. Within a platform, joins use IDs.

In [9]:
gr_books['title_join'] = normalize_title(gr_books['title'])
gr_reviews['title_join'] = normalize_title(gr_reviews['book_title'])
rt_movies['title_join'] = normalize_title(rt_movies['title'])
rt_audience['title_join'] = normalize_title(rt_audience['movie_title'])
rt_critics['title_join'] = normalize_title(rt_critics['movie_title'])

gr_books['genres_list'] = parse_genres(gr_books['genres'])
rt_movies['genres_list'] = parse_genres(rt_movies['genres'])

for label, df in [('Goodreads', gr_books), ('RT movies', rt_movies)]:
    counts = Counter(g for gl in df['genres_list'] for g in gl)
    no_genre = (df['genres_list'].str.len() == 0).sum()
    print(f"{label}: {len(counts)} distinct genres, {no_genre} titles with none. Top 10:")
    print('  ' + ', '.join(f'{g} ({n})' for g, n in counts.most_common(10)))

Goodreads: 753 distinct genres, 974 titles with none. Top 10:
  Fiction (2589), Audiobook (1630), Fantasy (1364), Romance (1194), Nonfiction (977), Young Adult (879), Contemporary (789), Adult (752), Classics (744), Mystery (635)
RT movies: 33 distinct genres, 1 titles with none. Top 10:
  Drama (1185), Comedy (921), Mystery & Thriller (799), Adventure (642), Action (609), Documentary (404), Fantasy (378), Horror (374), Animation (374), Crime (360)


## 9. Quality checks

These stop the notebook if the cleaned data breaks an assumption the analysis relies on.

In [10]:
def in_unit_range(s):
    s = s.dropna()
    return bool(((s >= 0) & (s <= 1)).all())

checks = {
    'books: one row per book_id':               gr_books['book_id'].is_unique,
    'GR reviews: one per (book, reviewer)':     not gr_reviews.duplicated(['book_id', 'reviewer_id']).any(),
    'GR reviews: every book_id is in books':    gr_reviews['book_id'].isin(gr_books['book_id']).all(),
    'RT movies: one row per movie_url':         rt_movies['movie_url'].is_unique,
    'RT reviews: every movie_url is in movies': (rt_audience['movie_url'].isin(rt_movies['movie_url']).all()
                                                 and rt_critics['movie_url'].isin(rt_movies['movie_url']).all()),
    'review ratings are within 0-1':            all(in_unit_range(df['rating_normalized']) for df in review_tables.values()),
    f'reviews have >= {MIN_WORDS} words':        all((df['word_count'] >= MIN_WORDS).all() for df in review_tables.values()),
    'review text is ASCII':                     all(df['review_text'].str.isascii().all() for df in review_tables.values()),
    'every review has a language label':        all(df['language'].notna().all() for df in review_tables.values()),
    'RT scores within 0-100':                   all(rt_movies[c].dropna().between(0, 100).all()
                                                    for c in ['tomatometer', 'audience_score']),
    'GR avg_rating within 1-5':                 gr_books['avg_rating'].between(1, 5).all(),
}
for name, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
failed = [name for name, ok in checks.items() if not ok]
assert not failed, f"Quality checks failed: {failed}"

  PASS  books: one row per book_id
  PASS  GR reviews: one per (book, reviewer)
  PASS  GR reviews: every book_id is in books
  PASS  RT movies: one row per movie_url
  PASS  RT reviews: every movie_url is in movies
  PASS  review ratings are within 0-1
  PASS  reviews have >= 5 words
  PASS  review text is ASCII
  PASS  every review has a language label
  PASS  RT scores within 0-100
  PASS  GR avg_rating within 1-5


In [11]:
outputs = {
    'cleaned_goodreads_books.csv': gr_books,
    'cleaned_goodreads_reviews.csv': gr_reviews,
    'cleaned_rotten_tomatoes_movies.csv': rt_movies,
    'cleaned_rotten_tomatoes_audience_reviews.csv': rt_audience,
    'cleaned_rotten_tomatoes_critic_reviews.csv': rt_critics,
}
before = dict(zip(outputs, raw_counts.values()))

summary = pd.DataFrame({
    'raw rows': before,
    'clean rows': {f: len(df) for f, df in outputs.items()},
    'columns': {f: df.shape[1] for f, df in outputs.items()},
})
summary['kept %'] = (summary['clean rows'] / summary['raw rows'] * 100).round(1)
summary

,raw rows,clean rows,columns,kept %
cleaned_goodreads_books.csv,6330,4997,19,78.9
cleaned_goodreads_reviews.csv,177261,135361,10,76.4
cleaned_rotten_tomatoes_movies.csv,3001,3001,27,100.0
cleaned_rotten_tomatoes_audience_reviews.csv,15988,15264,11,95.5
cleaned_rotten_tomatoes_critic_reviews.csv,25581,25136,14,98.3


## 10. Save

In [12]:
os.makedirs(CLEAN_DIR, exist_ok=True)
for fname, df in outputs.items():
    df.to_csv(os.path.join(CLEAN_DIR, fname), index=False)

print(f"Saved {len(outputs)} cleaned datasets to {CLEAN_DIR}/")
for f in sorted(os.listdir(CLEAN_DIR)):
    size_mb = os.path.getsize(os.path.join(CLEAN_DIR, f)) / 1e6
    print(f"  {f:48s} {size_mb:7.1f} MB")

Saved 5 cleaned datasets to data/cleaned/
  cleaned_goodreads_books.csv                          6.5 MB
  cleaned_goodreads_reviews.csv                      253.8 MB
  cleaned_rotten_tomatoes_audience_reviews.csv         6.2 MB
  cleaned_rotten_tomatoes_critic_reviews.csv           7.6 MB
  cleaned_rotten_tomatoes_movies.csv                   2.1 MB
